# Pixel tolerance expressed as fraction of laser spot

Paper reference: §3.2 (System Model), §3.3.6 (Post-processing).

Combines the pixel-sensitivity analysis of
`pixel_sensitivity.ipynb` with the apparent laser-spot geometry used
in `laser_spot_path.ipynb`. At each target distance $z$, it expresses
the 1/5/10/15/20 % depth-error pixel distance as a fraction of the
laser dot's width (or height) in pixels.

A value of 100 % means a labeler could place the label anywhere inside
the visible laser blob and still stay within that error threshold — a
practical bound on how accurate field labeling needs to be relative to
the spot the labeler actually sees.

No paper figure; diagnostic only.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

from fishsense_imwut.camera import reconstruct_points
from fishsense_imwut.constants import (
    FOCAL_LENGTH_PX,
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    LASER_DIAMETER_M,
)

## Setup

In [ ]:
camera_intrinsics = np.array([
    [FOCAL_LENGTH_PX, 0, IMAGE_WIDTH / 2],
    [0, FOCAL_LENGTH_PX, IMAGE_HEIGHT / 2],
    [0, 0, 1],
])
inverted_camera_intrinsics = np.linalg.inv(camera_intrinsics)

laser_position = np.array([-0.04, -0.11, 0])
laser_direction = np.array([1e-10, 1e-10, 1])

In [ ]:
STEP_COUNT = 1000
t = np.linspace(0.5, 30, STEP_COUNT)

p = laser_position[:, np.newaxis] + t[np.newaxis, :] * laser_direction[:, np.newaxis]
s = camera_intrinsics @ (p / p[2, :])
s_pixel = np.round(s)

## Apparent laser-spot size

Back-project four cardinal points around each laser dot center
(±`LASER_DIAMETER_M / 2` in $x$/$y$) and reproject them. The pixel
width $(s_\text{right} - s_\text{left})$ and height $(s_\text{up} -
s_\text{down})$ give the apparent spot extent on the image plane at
each distance.

In [ ]:
p_left  = p.copy(); p_left[0, :]  -= LASER_DIAMETER_M / 2
p_right = p.copy(); p_right[0, :] += LASER_DIAMETER_M / 2
p_up    = p.copy(); p_up[1, :]    += LASER_DIAMETER_M / 2
p_down  = p.copy(); p_down[1, :]  -= LASER_DIAMETER_M / 2

s_left  = camera_intrinsics @ (p_left  / p_left[2, :])
s_right = camera_intrinsics @ (p_right / p_right[2, :])
s_up    = camera_intrinsics @ (p_up    / p_up[2, :])
s_down  = camera_intrinsics @ (p_down  / p_down[2, :])

## Sweep

In [ ]:
x_range = np.arange(0, IMAGE_WIDTH)
y_range = np.arange(0, IMAGE_HEIGHT)
xy_grid = np.array(np.meshgrid(x_range, y_range, indexing='ij'))
xy_homogeneous = np.vstack((
    xy_grid.reshape(2, -1),
    np.ones((1, xy_grid.shape[1] * xy_grid.shape[2])),
))

thresholds = [1, 5, 10, 15, 20]
error_distances = {k: [] for k in thresholds}
z_values, widths, heights = [], [], []

for idx in tqdm(range(s.shape[1])):
    z = p[2, idx]
    if z < 0.5:
        continue

    width  = s_right[0, idx] - s_left[0, idx]
    height = s_up[1, idx]    - s_down[1, idx]

    p_reconstructed, _ = reconstruct_points(
        xy_homogeneous, inverted_camera_intrinsics, laser_position, laser_direction,
    )
    percent_errors = np.abs(p_reconstructed[2, :] - z) / z * 100
    distances = np.linalg.norm(
        xy_homogeneous[:2, :] - s_pixel[:2, idx:idx + 1], axis=0,
    )

    for k in thresholds:
        above = distances[percent_errors > k]
        error_distances[k].append(above.min() if above.size else np.nan)

    z_values.append(z)
    widths.append(width)
    heights.append(height)

    if z > 5:
        break

z_values = np.array(z_values)
widths   = np.array(widths)
heights  = np.array(heights)
for k in thresholds:
    error_distances[k] = np.array(error_distances[k])

In [ ]:
fig, (ax_w, ax_h) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for k in thresholds:
    ax_w.plot(z_values, error_distances[k] / widths * 100,  label=f'{k}% error')
    ax_h.plot(z_values, error_distances[k] / heights * 100, label=f'{k}% error')

for ax, label in zip((ax_w, ax_h), ('spot width', 'spot height')):
    ax.set_xlabel('Target distance $z$ (m)')
    ax.set_ylabel(f'Pixel tolerance (% of {label})')
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend()
ax_w.set_title('Tolerance as % of spot width')
ax_h.set_title('Tolerance as % of spot height')
fig.tight_layout()
fig

Where a curve sits at or above 100 %, the pixel-distance tolerance for that error threshold is larger than the apparent laser spot — a labeler dropping the label anywhere inside the visible blob stays within the threshold. Where it drops below 100 %, sub-spot precision is required and detector/annotator accuracy starts to matter.